# Notebook 05b — Population Clustering Refinement: Adult-Only K-Prototypes

This notebook refines the clustering strategy established in Notebook 05 based on two decisions made during portfolio review:

1. **Exclude minors** — clusters 2 and 5 in the original k=7 solution represented dependent children and were not targetable in a B2C context. The clustering input is restricted to adults (18+) to ensure all resulting segments are actionable.
2. **Replace PUMA with area_type** — raw PUMA codes introduced high cardinality noise at national scale. A simplified 3-category area type (urban / suburban / rural) is derived from PUMA and used instead.

Based on the algorithm evaluation in Notebook 06, **K-Prototypes** was selected as the final clustering approach. This notebook runs K-Prototypes for **k = 6, 7, and 8** only — the candidate range identified as optimal — rather than repeating the full k=4–15 experiment.

## Objectives

- Filter the structural population dataset to adults (18+)
- Derive `area_type` from PUMA as a lower-cardinality geographic signal
- Prepare the K-Prototypes feature matrix with the revised feature set
- Run K-Prototypes for k = 6, 7, 8 and save models and results
- Export cluster assignments for evaluation in Notebook 06b

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# This notebook lives at Market_Kinetics/notebooks/
# parents[0] = notebooks/
# parents[1] = Market_Kinetics/  ← project root

PROJECT_ROOT = Path().resolve().parent
DATA_PATH    = PROJECT_ROOT / "data" / "societal_interim" / "acs_pums_5y"
MODELS_DIR   = PROJECT_ROOT / "data" / "societal_models"
FILE_NAME    = "mk_structural_population_layer_v1.parquet"

print("=" * 80)
print("PHASE 1: DATA PREPARATION & PREPROCESSING STRATEGY (ADULT-ONLY RERUN)")
print("=" * 80)
print(f"\nProject root : {PROJECT_ROOT}")
print(f"Data path    : {DATA_PATH}")
print(f"Models dir   : {MODELS_DIR}")

PHASE 1: DATA PREPARATION & PREPROCESSING STRATEGY (ADULT-ONLY RERUN)

Project root : /Users/marcomagnolo/Projects/Market_Kinetics
Data path    : /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y
Models dir   : /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_models


## Step 1: Load the Merged Structural Population Dataset

Load `mk_structural_population_layer_v1.parquet` with 19 columns representing the merged household (HUS) + person (PUS) structural features.

In [2]:
# Load the dataset
df = pd.read_parquet(DATA_PATH / FILE_NAME)

print("\n" + "=" * 80)
print("STEP 1: LOAD MERGED STRUCTURAL POPULATION DATASET")
print("=" * 80)

print(f"\n✓ Loaded: {FILE_NAME}")
print(f"  Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n  Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nData types:")
print(df.dtypes)


STEP 1: LOAD MERGED STRUCTURAL POPULATION DATASET

✓ Loaded: mk_structural_population_layer_v1.parquet
  Dataset shape: 1,000,000 rows × 19 columns

  Columns: ['serialno', 'sporder', 'pwgtp', 'actor_class', 'age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed', 'income_tier_pct', 'mar_tier', 'commute_tier', 'tenure', 'household_size', 'vehicle_count', 'puma', 'hhincome_tier', 'household_type']

First 5 rows:
        serialno  sporder  pwgtp actor_class age_bin sex_label  race_eth  \
0  2023HU1043211        2     58       Adult   35-44      Male  Black_NH   
1  2019HU1076190        2     46       Adult   35-44      Male  Black_NH   
2  2019GQ0046130        1     12       Adult   35-44      Male  Black_NH   
3  2019HU0403832        1     76       Adult   35-44      Male  Black_NH   
4  2019HU0277198        1     64       Adult   35-44      Male  Black_NH   

     edu_tier                  emp_tier income_tier_fixed income_tier_pct  \
0  HS_or_less             

In [3]:
# Preview age_bin values — needed to define the adult filter in the next step
print(f"\nage_bin unique values (sorted):")
print(sorted(df['age_bin'].dropna().unique()))


age_bin unique values (sorted):
['0-5', '13-17', '18-24', '25-34', '35-44', '45-54', '55-64', '6-12', '65+']


## Step 2: Inspect All 19 Columns

Examine summary statistics, missing values, and distributions for each column.

In [4]:

from pandas.api.types import (
    is_object_dtype,
    is_string_dtype,
    is_categorical_dtype,
    is_numeric_dtype
)

print("\n" + "=" * 80)
print("STEP 2: INSPECT ALL 19 COLUMNS")
print("=" * 80)

# Basic info
print("\nBasic Info:")
df.info()

# Missing values
print("\nMissing Values:")
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": df.isna().sum().values,
    "Missing %": (df.isna().sum().values / len(df) * 100).round(2)
})
print(missing_summary.to_string(index=False))

# Unique values per column
print("\nUnique Values per Column:")
for col in df.columns:
    n_unique = df[col].nunique(dropna=True)
    dtype_str = str(df[col].dtype)
    print(f"  {col:25s} | Type: {dtype_str:12s} | Unique: {n_unique:6d}")

# Detailed examination of each column
print("\nDetailed Column Inspection:")
for col in df.columns:
    s = df[col]
    print(f"\n  {col}:")
    print(f"    Type: {s.dtype}")
    print(f"    Missing: {s.isna().sum()} ({s.isna().sum()/len(df)*100:.2f}%)")

    if is_object_dtype(s) or is_string_dtype(s) or is_categorical_dtype(s):
        vc = s.value_counts(dropna=False)
        print(f"    Unique values: {s.nunique(dropna=True)}")
        print(f"    Top 5: {vc.head().to_dict()}")
    elif is_numeric_dtype(s):
        print(f"    Min: {s.min()}, Max: {s.max()}, Mean: {s.mean():.2f}, Median: {s.median():.2f}")
    else:
        print("    Unhandled dtype - inspect manually")


STEP 2: INSPECT ALL 19 COLUMNS

Basic Info:
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   serialno           1000000 non-null  str     
 1   sporder            1000000 non-null  int64   
 2   pwgtp              1000000 non-null  int64   
 3   actor_class        1000000 non-null  str     
 4   age_bin            1000000 non-null  string  
 5   sex_label          1000000 non-null  string  
 6   race_eth           1000000 non-null  string  
 7   edu_tier           1000000 non-null  string  
 8   emp_tier           804465 non-null   str     
 9   income_tier_fixed  1000000 non-null  string  
 10  income_tier_pct    778466 non-null   str     
 11  mar_tier           1000000 non-null  string  
 12  commute_tier       473674 non-null   category
 13  tenure             1000000 non-null  string  
 14  household_size     1000000 non-nu

## Step 3: Define Clustering Feature Set

Select 14 features for clustering. Exclude identifiers (`serialno`, `sporder`) and weight column (`pwgtp`).

In [5]:
print("\n" + "=" * 80)
print("STEP 2: FILTER TO ADULTS")
print("=" * 80)

# ── Adult filter ──────────────────────────────────────────────────────────────
df_adults = df[df['actor_class'] == 'Adult'].copy()

print(f"\n✓ Adult filter applied:")
print(f"  Original : {len(df):,} records")
print(f"  Adults   : {len(df_adults):,} records")
print(f"  Minors   : {len(df) - len(df_adults):,} removed ({(len(df) - len(df_adults)) / len(df) * 100:.1f}%)")


print("\n" + "=" * 80)
print("STEP 3: DEFINE CLUSTERING FEATURE SET")
print("=" * 80)

# Changes from notebook 05:
#   REMOVED  commute_tier    — 52.6% missing, unreliable signal
#   REMOVED  puma            — high cardinality, no reliable area-type mapping
#                              at PUMA level without external RUCA/Census lookup.
#                              To be added in a future iteration.
#   REMOVED  income_tier_pct — overlaps with income_tier_fixed, 22% missing
# Changes from v1:
#   REMOVED  hhincome_tier   — household-level feature; targeting unit is the
#                              individual. Kept as descriptor on archetype card.
clustering_features = [
    'age_bin',
    'sex_label',
    'race_eth',
    'edu_tier',
    'emp_tier',
    'income_tier_fixed',
    'mar_tier',
    'tenure',
    'household_size',
    'vehicle_count',
    'household_type',
]

exclude_features = [
    'serialno', 'sporder', 'pwgtp', 'actor_class',
    'puma', 'commute_tier', 'income_tier_pct',
    'hhincome_tier',   # household income — kept as archetype descriptor, not a clustering feature
]

print(f"\n✓ Clustering features ({len(clustering_features)}):")
for i, feat in enumerate(clustering_features, 1):
    if feat in df_adults.columns:
        print(f"  {i:2d}. {feat:25s} ✓")
    else:
        print(f"  {i:2d}. {feat:25s} ✗ NOT FOUND")

print(f"\n✓ Excluded features ({len(exclude_features)}):")
for feat in exclude_features:
    print(f"   - {feat}")

missing = [f for f in clustering_features if f not in df_adults.columns]
if missing:
    print(f"\n⚠ WARNING — missing features: {missing}")
else:
    print(f"\n✓ All {len(clustering_features)} features present")

# Keep serialno, sporder, pwgtp alongside for weighting and traceability
df_cluster = df_adults[clustering_features + ['serialno', 'sporder', 'pwgtp']].copy()
print(f"\n✓ Clustering DataFrame shape: {df_cluster.shape}")
print(f"  {df_cluster.shape[0]:,} adults × {df_cluster.shape[1]} columns")


STEP 2: FILTER TO ADULTS

✓ Adult filter applied:
  Original : 1,000,000 records
  Adults   : 778,466 records
  Minors   : 221,534 removed (22.2%)

STEP 3: DEFINE CLUSTERING FEATURE SET

✓ Clustering features (11):
   1. age_bin                   ✓
   2. sex_label                 ✓
   3. race_eth                  ✓
   4. edu_tier                  ✓
   5. emp_tier                  ✓
   6. income_tier_fixed         ✓
   7. mar_tier                  ✓
   8. tenure                    ✓
   9. household_size            ✓
  10. vehicle_count             ✓
  11. household_type            ✓

✓ Excluded features (8):
   - serialno
   - sporder
   - pwgtp
   - actor_class
   - puma
   - commute_tier
   - income_tier_pct
   - hhincome_tier

✓ All 11 features present

✓ Clustering DataFrame shape: (778466, 14)
  778,466 adults × 14 columns


## Step 4: Handle Missing Values

Treat missing values as explicit "Missing" category for categorical features. Document the strategy for each feature.

In [6]:
print("\n" + "=" * 80)
print("STEP 4: HANDLE MISSING VALUES")
print("=" * 80)

# ── Feature type classification ───────────────────────────────────────────────
categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "household_type",
    "tenure",
]

numeric_features = [
    "household_size",
    "vehicle_count",
]

print("\nFeature Type Classification:")
print(f"  Categorical ({len(categorical_features)}): {categorical_features}")
print(f"  Numeric     ({len(numeric_features)}): {numeric_features}")

# ── Tenure mapping ────────────────────────────────────────────────────────────
# Raw ACS codes → readable labels (same as notebook 05)
tenure_map = {
    "1": "Owner",
    "2": "Renter",
    "3": "No_Rent",
    "4": "Other",
    "group_quarters": "Group_Quarters",
}
df_cluster["tenure"] = df_cluster["tenure"].astype(str).map(tenure_map).fillna("Other")
print("\n✓ Tenure mapped to readable labels")
print(f"  Values: {sorted(df_cluster['tenure'].unique())}")

# ── Missing value analysis ────────────────────────────────────────────────────
print("\nMissing Value Analysis by Feature:")

for feat in clustering_features:
    n_missing = df_cluster[feat].isna().sum()
    pct_missing = n_missing / len(df_cluster) * 100
    if n_missing == 0:
        strategy = "Complete — no action needed"
    elif feat in categorical_features:
        strategy = "Fill NaN → 'Missing' category"
    else:
        strategy = "Handle per algorithm"
    print(f"  {feat:25s} | Missing: {n_missing:6d} ({pct_missing:5.1f}%) | {strategy}")

# ── Apply missing value strategy ──────────────────────────────────────────────
df_clean = df_cluster.copy()

for feat in categorical_features:
    if df_clean[feat].isna().any():
        df_clean[feat] = df_clean[feat].astype("string").fillna("Missing")

filled = [f for f in categorical_features if df_cluster[f].isna().any()]
print(f"\n✓ Categorical NaN filled with 'Missing': {filled if filled else 'none — all complete'}")

numeric_with_na = [f for f in numeric_features if df_cluster[f].isna().any()]
print(f"✓ Numeric features with NA (handled per algorithm): {numeric_with_na if numeric_with_na else 'none'}")

# ── Verification ──────────────────────────────────────────────────────────────
print("\nVerification — remaining NaN in df_clean (clustering features only):")
remaining = df_clean[clustering_features].isna().sum()
remaining = remaining[remaining > 0]
if remaining.empty:
    print("  ✓ No remaining NaN in any clustering feature")
else:
    print(remaining)


STEP 4: HANDLE MISSING VALUES

Feature Type Classification:
  Categorical (9): ['age_bin', 'sex_label', 'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed', 'mar_tier', 'household_type', 'tenure']
  Numeric     (2): ['household_size', 'vehicle_count']

✓ Tenure mapped to readable labels
  Values: ['Group_Quarters', 'No_Rent', 'Other', 'Owner', 'Renter']

Missing Value Analysis by Feature:
  age_bin                   | Missing:      0 (  0.0%) | Complete — no action needed
  sex_label                 | Missing:      0 (  0.0%) | Complete — no action needed
  race_eth                  | Missing:      0 (  0.0%) | Complete — no action needed
  edu_tier                  | Missing:      0 (  0.0%) | Complete — no action needed
  emp_tier                  | Missing:      0 (  0.0%) | Complete — no action needed
  income_tier_fixed         | Missing:      0 (  0.0%) | Complete — no action needed
  mar_tier                  | Missing:      0 (  0.0%) | Complete — no action needed
  tenure

## Step 6: Encoding Plan for K-Prototypes Clustering

**Algorithm**: K-Prototypes handles both categorical and numeric features simultaneously.

**Strategy**:
1. Categorical features: Keep as category labels with "Missing" as explicit category
2. Numeric features: Keep as continuous values (handle NaN via imputation or median)
3. Mixed representation: Combine categorical + numeric in single feature matrix

In [7]:
print("\n" + "=" * 80)
print("STEP 5: K-PROTOTYPES ENCODING PLAN")
print("=" * 80)

# Changes from notebook 05:
#   REMOVED commute_tier — 52.6% missing
#   REMOVED puma         — high cardinality, no reliable area-type proxy
# Changes from v1:
#   REMOVED hhincome_tier — household-level feature; kept as archetype descriptor only

categorical_features = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "mar_tier",
    "tenure",
    "household_type",
]

numeric_features = [
    "household_size",
    "vehicle_count",
]

# ── Feature split inspection ──────────────────────────────────────────────────
print("\n✓ K-Prototypes: Feature Split")

print(f"\n  Categorical ({len(categorical_features)}):")
for feat in categorical_features:
    n_cats = df_clean[feat].nunique(dropna=True)
    print(f"    - {feat:25s} | Categories: {n_cats}")

print(f"\n  Numeric ({len(numeric_features)}):")
for feat in numeric_features:
    min_val  = df_clean[feat].min()
    max_val  = df_clean[feat].max()
    mean_val = df_clean[feat].mean()
    n_miss   = df_clean[feat].isna().sum()
    print(f"    - {feat:25s} | Range: [{min_val:.0f}, {max_val:.0f}] "
          f"| Mean: {mean_val:.2f} | Missing: {n_miss}")

# ── Numeric handling ──────────────────────────────────────────────────────────
print("\n✓ Numeric Feature Handling:")
for feat in numeric_features:
    median_val = df_clean[feat].median()
    print(f"    - {feat:25s} | Impute NaN → median ({median_val:.1f})")

# ── Build K-Prototypes matrix ─────────────────────────────────────────────────
X_kprototypes = df_clean[categorical_features + numeric_features].copy()

for col in categorical_features:
    X_kprototypes[col] = X_kprototypes[col].astype(str)

for col in numeric_features:
    X_kprototypes[col] = pd.to_numeric(X_kprototypes[col], errors="coerce")
    X_kprototypes[col] = X_kprototypes[col].fillna(X_kprototypes[col].median())

X_kprototypes["household_size"] = X_kprototypes["household_size"].astype("float64")
X_kprototypes["vehicle_count"]  = X_kprototypes["vehicle_count"].astype("float64")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n✓ K-Prototypes matrix built:")
print(f"  Shape      : {X_kprototypes.shape}")
print(f"  Categorical: {len(categorical_features)} features")
print(f"  Numeric    : {len(numeric_features)} features")

print("\nDtypes:")
print(X_kprototypes.dtypes)

print("\nHead:")
print(X_kprototypes.head())

# ── Categorical column indices (needed by KPrototypes at fit time) ─────────────
cat_col_indices = [
    X_kprototypes.columns.get_loc(col)
    for col in categorical_features
]
print(f"\n✓ Categorical column indices: {cat_col_indices}")


STEP 5: K-PROTOTYPES ENCODING PLAN

✓ K-Prototypes: Feature Split

  Categorical (9):
    - age_bin                   | Categories: 6
    - sex_label                 | Categories: 2
    - race_eth                  | Categories: 5
    - edu_tier                  | Categories: 4
    - emp_tier                  | Categories: 5
    - income_tier_fixed         | Categories: 6
    - mar_tier                  | Categories: 3
    - tenure                    | Categories: 5
    - household_type            | Categories: 2

  Numeric (2):
    - household_size            | Range: [1, 20] | Mean: 2.93 | Missing: 0
    - vehicle_count             | Range: [0, 6] | Mean: 2.06 | Missing: 0

✓ Numeric Feature Handling:
    - household_size            | Impute NaN → median (2.0)
    - vehicle_count             | Impute NaN → median (2.0)

✓ K-Prototypes matrix built:
  Shape      : (778466, 11)
  Categorical: 9 features
  Numeric    : 2 features

Dtypes:
age_bin                  str
sex_label          

## Summary: Phase 1 Complete

### Dataset Loaded and Filtered ✓
- **File**: mk_structural_population_layer_v1.parquet
- **Original dimensions**: 1,000,000 records × 19 columns
- **After adult filter**: 778,466 records (minors removed via `actor_class == 'Adult'`)

### Feature Set Defined ✓
- **12 features selected** (down from 14 in notebook 05)
- **Removed**: `commute_tier` (52.6% missing), `puma` (high cardinality, no reliable area-type proxy)
- **Removed**: `income_tier_pct` (overlaps with `income_tier_fixed`, 22% missing)
- **Tenure**: mapped from raw ACS codes to readable labels (Owner, Renter, No_Rent, Other, Group_Quarters)

### Feature Matrix Built ✓
- **X_kprototypes**: 778,466 × 12 — mixed categorical (10) + numeric (2)
- No missing values remaining
- Categorical column indices: [0–9]

### Next Steps (Phase 2)
- Run K-Prototypes for k = 6, 7, 8
- Save models and cluster assignments for evaluation in notebook 06b

# Phase 2 — K-Prototypes Clustering

Run K-Prototypes for k = 6, 7, and 8 on the adult-only feature matrix.
Based on the full algorithm evaluation in notebook 05/06, K-Prototypes was
selected as the final approach. This phase runs the winning algorithm only,
across the three candidate k values identified in notebook 06.

In [8]:
from kmodes.kprototypes import KPrototypes

print("\n" + "=" * 80)
print("PHASE 2: K-PROTOTYPES CLUSTERING")
print("=" * 80)


PHASE 2: K-PROTOTYPES CLUSTERING


In [9]:
kproto_results = []
kproto_models  = {}

# Use cat_col_indices computed in Step 5 — guaranteed to match X_kprototypes column order
for k in [6, 7, 8]:
    print(f"\nFitting K-Prototypes k={k}...")
    model = KPrototypes(
        n_clusters=k,
        init="Huang",
        n_init=5,
        verbose=0,
        random_state=42,
    )
    clusters = model.fit_predict(X_kprototypes, categorical=cat_col_indices)
    kproto_models[k] = model

    unique, counts = np.unique(clusters, return_counts=True)
    sizes = dict(zip(unique.tolist(), counts.tolist()))

    kproto_results.append({
        "algorithm":       "K-Prototypes",
        "k":               k,
        "cost":            model.cost_ if hasattr(model, "cost_") else np.nan,
        "cluster_sizes":   sizes,
    })

    print(f"  ✓ k={k} | cost: {model.cost_:.0f} | sizes: {sizes}")

print("\n✓ K-Prototypes runs complete.")


Fitting K-Prototypes k=6...
  ✓ k=6 | cost: 3026711 | sizes: {0: 181239, 1: 113665, 2: 149850, 3: 39741, 4: 162815, 5: 131156}

Fitting K-Prototypes k=7...
  ✓ k=7 | cost: 2901295 | sizes: {0: 122182, 1: 134581, 2: 38470, 3: 118386, 4: 115407, 5: 91685, 6: 157755}

Fitting K-Prototypes k=8...
  ✓ k=8 | cost: 2851201 | sizes: {0: 168911, 1: 58701, 2: 129338, 3: 77132, 4: 69317, 5: 184459, 6: 71713, 7: 18895}

✓ K-Prototypes runs complete.


## Display Results

Combine results into one DataFrame and print summaries.

In [10]:
results = pd.DataFrame(kproto_results)

print("\n✓ K-Prototypes results summary:")
print(results[["algorithm", "k", "cost"]].to_string(index=False))

print("\nCluster size distributions:")
for _, row in results.iterrows():
    print(f"  k={row['k']} | sizes: {row['cluster_sizes']}")

print("\n✓ Phase 2 complete.")


✓ K-Prototypes results summary:
   algorithm  k         cost
K-Prototypes  6 3.026711e+06
K-Prototypes  7 2.901295e+06
K-Prototypes  8 2.851201e+06

Cluster size distributions:
  k=6 | sizes: {0: 181239, 1: 113665, 2: 149850, 3: 39741, 4: 162815, 5: 131156}
  k=7 | sizes: {0: 122182, 1: 134581, 2: 38470, 3: 118386, 4: 115407, 5: 91685, 6: 157755}
  k=8 | sizes: {0: 168911, 1: 58701, 2: 129338, 3: 77132, 4: 69317, 5: 184459, 6: 71713, 7: 18895}

✓ Phase 2 complete.


In [11]:
import joblib

print("\n" + "=" * 80)
print("PHASE 3: SAVE MODELS AND RESULTS")
print("=" * 80)

# ── Save to societal_models/ alongside notebook 05 outputs ───────────────────
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save K-Prototypes models
for k, model in kproto_models.items():
    path = MODELS_DIR / f"kprototypes_v2_k{k}.joblib"
    joblib.dump(model, path)
    print(f"  ✓ Saved kprototypes_v2_k{k}.joblib")

print(f"\n✓ Saved {len(kproto_models)} K-Prototypes models to {MODELS_DIR}")

# Save results summary
results_path = MODELS_DIR / "clustering_experiment_results_v2.csv"
results.to_csv(results_path, index=False)
print(f"✓ Saved results summary to {results_path}")

# Save X_kprototypes and cat_col_indices for use in notebook 06b
joblib.dump(X_kprototypes,   MODELS_DIR / "X_kprototypes_v2.joblib")
joblib.dump(cat_col_indices, MODELS_DIR / "cat_col_indices_v2.joblib")
joblib.dump(df_cluster,      MODELS_DIR / "df_cluster_v2.joblib")
print(f"✓ Saved feature matrix, column indices, and cluster dataframe")

print("\n✓ Phase 3 complete. Ready for evaluation in notebook 06b.")


PHASE 3: SAVE MODELS AND RESULTS
  ✓ Saved kprototypes_v2_k6.joblib
  ✓ Saved kprototypes_v2_k7.joblib
  ✓ Saved kprototypes_v2_k8.joblib

✓ Saved 3 K-Prototypes models to /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_models
✓ Saved results summary to /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_models/clustering_experiment_results_v2.csv
✓ Saved feature matrix, column indices, and cluster dataframe

✓ Phase 3 complete. Ready for evaluation in notebook 06b.
